In [1]:
import pandas as pd
import re
import json

TSV to CSV

In [2]:
def read_data(file_path):
    """
    Reads the annotated data and parses it into a desired structured Df.

    """
    data = []
    current_text = ""
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()
            if line.startswith('#Text='):
                current_text = line.split('=')[1].strip()  # Extract the text following this prefix.
            elif line and not line.startswith('#'):
                parts = line.split('\t')
                if len(parts) >= 4:
                    sentence_id, token_id = parts[0].split('-')
                    if len(sentence_id) == 0 or len(token_id) == 0:
                        continue  # Skip entries without proper ID formatting.

                    # The labels might have annotations like Object[1], split and clean them
                    spo_labels = [re.sub(r'\[\d+\]', '', label) for label in parts[3].split('|')]
                    while len(spo_labels) < 3:
                        spo_labels.append('_')

                    # Replace '-' with 'other' in spo_label_1
                    spo_label_1 = 'other' if spo_labels[0] == '_' else spo_labels[0]

                    token_data = {
                        'sentence_id': sentence_id,
                        'token_id': token_id,
                        'sentence': current_text,
                        'token': parts[2],
                        'spo_label_1': spo_label_1, 
                        'spo_label_2': spo_labels[1] if len(spo_labels) > 1 else '_',
                        'spo_label_3': spo_labels[2] if len(spo_labels) > 2 else '_'
                    }
                    data.append(token_data)
                else:
                    print(f"Skipping malformed line in {file_path}: {line}")

    return pd.DataFrame(data)


def aggregate_annotations(data):
    """
    Aggregates tokens and labels by 'annotator_id' and 'sentence_id'.
    
    """
    def concat_items(series):
        """
        Takes a pandas series object and converts it into a list.

        """
        return list(series)
    
    # Aggregate tokens and SPO-labels by 'annotator_id' and 'sentence_id'
    grouped = data.groupby(['sentence_id'], sort=False)

    return grouped.agg({
        'sentence': 'first',
        'token': concat_items,
        'spo_label_1': concat_items,
        'spo_label_2': concat_items,
        'spo_label_3': concat_items
    })


def combine_labels(row):
    """
    Combines the SPO-labels into a single string for each token.
    
    """
    spo_labels = []
    for i in range(len(row['token'])):
        # Collect all SPO-labels 
        spo_labels_list = [row['spo_label_1'][i], row['spo_label_2'][i], row['spo_label_3'][i]]
        # Remove placeholder '_' and keep duplicates
        spo_labels_filtered = [label for label in spo_labels_list if label != '_']
        # Join the SPO-labels with ';'
        spo_labels.append(';'.join(spo_labels_filtered))

    return spo_labels


def prepare_final_dataframe(aggregated):
    """
    Cleans and restructures the data into a Df ready for further analysis.
    
    """
    # Combines the SPO-labels and store them in a list
    spo_labels_list = aggregated.apply(combine_labels, axis=1)

    # Assign the list as a new column
    aggregated['spo_labels'] = spo_labels_list

    # Rename columns and drop unused ones
    final_data = aggregated.rename(columns={
        'token': 'tokens',
        'text': 'sentence',
    }).drop(columns=['spo_label_1', 'spo_label_2', 'spo_label_3'])

    # Reset the index and keep the old index columns as regular columns
    final_data.reset_index(inplace=True, drop=False)

    # Replace empty strings in lists with 'None'
    final_data['spo_labels'] = final_data['spo_labels'].apply(
        lambda x: [tag if tag.strip() else 'other' for tag in x])

    return final_data


validation_data = read_data('C:/Users/ntanavarass/Desktop/Conversational-Triple-Extraction/data annotation/annotated data/annotated_validation_data.tsv')
validation_data.to_csv('C:/Users/ntanavarass/Desktop/Conversational-Triple-Extraction/final data/final_validation_data.csv', index=False)

test_data = read_data('C:/Users/ntanavarass/Desktop/Conversational-Triple-Extraction/data annotation/annotated data/annotator1.tsv')
test_data.to_csv('C:/Users/ntanavarass/Desktop/Conversational-Triple-Extraction/final data/final_test_data.csv', index=False)

# aggregated_test_data = aggregate_annotations(test_data)
# final_test_data = prepare_final_dataframe(aggregated_test_data)
# final_test_data.to_csv('C:/Users/ntanavarass/Desktop/Hybrid-AI-for-Diabetes-Healthcare-Management-1/final data/test_data.csv', index=False)

In [ ]:
annotated_test_data = pd.read_csv('C:/Users/ntanavarass/Desktop/Hybrid-AI-for-Diabetes-Healthcare-Management-1/final data/test_data.csv')

with open(r"C:/Users/ntanavarass/Desktop/Hybrid-AI-for-Diabetes-Healthcare-Management-1/data preprocessing/data/initial_test_data.txt", "r") as file:
    initial_test_data = file.readlines()



conversations = []
utterances = []
conversation_id = 1
utterance_id = 1

current_conversation = []

for line in initial_test_data:
    # Check for blank lines to determine the end of a conversation
    if line.strip() == "":
        if current_conversation:
            # Save the current conversation and reset for the next
            conversations.append((conversation_id, current_conversation))
            current_conversation = []
        conversation_id += 1
        utterance_id = 1
        continue

    # Add the current utterance to the ongoing conversation
    current_conversation.append((utterance_id, line.strip()))
    utterances.append({
        'conversation_id': conversation_id,
        'utterance_id': utterance_id,
        'utterance': line.strip()
    })
    utterance_id += 1

# Add the last conversation if it was not added
if current_conversation:
    conversations.append((conversation_id, current_conversation))

# Create a DataFrame for the identified utterances
utterances_df = pd.DataFrame(utterances)

# List to store the final structured data with correct conversation and utterance assignments
merged_data = []

# Iterate over each sentence in the test data to find its corresponding utterance and conversation
for index, row in annotated_test_data.iterrows():
    sentence_text = row['sentence'].strip()
    conv_id = None
    utt_id = None

    # Find the correct conversation and utterance by checking for partial matches to allow for slight variations
    for conv_id, conversation in conversations:
        for utt_id, utterance in conversation:
            # Use a relaxed match that checks if the sentence is partially found within the utterance
            if sentence_text in utterance or utterance in sentence_text:
                # Assign the correct conversation and utterance ids
                merged_row = {
                    'conversation_id': conv_id,
                    'utterance_id': utt_id,
                    'sentence_id': row['sentence_id'],
                    'sentence': row['sentence'],
                    'tokens': row['tokens'],
                    'spo_labels': row['spo_labels']
                }
                merged_data.append(merged_row)
                break
        if conv_id and utt_id:
            break


# Create the final corrected test set 
final_test_set = pd.DataFrame(merged_data)
final_test_set.to_csv('C:/Users/ntanavarass/Desktop/Hybrid-AI-for-Diabetes-Healthcare-Management-1/final data/final_test_set.csv', index=False)

JSON to CSV

In [4]:
# Transform the JSON file into a CSV
annotated_train_data = 'C:/Users/ntanavarass/Desktop/Hybrid-AI-for-Diabetes-Healthcare-Management-1/data annotation/corrected_final_train_data.json'

with open(annotated_train_data, 'r', encoding='utf-8') as file:
    data = file.read()

train_data = json.loads(data)
df = pd.json_normalize(train_data)

# Since 'sentence' repeats, keep only the first occurrence with each 'sentence_id'
df['sentence'] = df.groupby('sentence_id')['sentence'].transform('first')
df.drop_duplicates(subset=['sentence_id', 'token_id'], inplace=True)

# Select and rename columns as necessary
df_final = df[['sentence_id', 'token_id', 'sentence', 'token', 'spo_label_1', 'spo_label_2', 'spo_label_3']]

# Export to CSV
df_final.to_csv('C:/Users/ntanavarass/Desktop/Hybrid-AI-for-Diabetes-Healthcare-Management-1/final data/final_train_data.csv', index=False)